# TASK 2.1: XÂY DỰNG BỘ KIỂM THỬ SUY BIẾN VẬT LÝ (DEGRADED TEST SET)
### PIPELINE TỰ ĐỘNG SINH 6 BIẾN THỂ SUY BIẾN VẬT LÝ & ĐÓNG GÓI CHUẨN AN TOÀN LÊN GOOGLE DRIVE 5TB

> **Người thực hiện**: Thành viên A *(Data & Infra Lead)*  
> **Môi trường**: Google Colab CPU đa nhân / GPU T4  
> **Hạ tầng dữ liệu**: Google Drive 5TB (`/content/drive/MyDrive/Fatformer`)  
> **Đầu ra nghiệm thu**: Tệp nén `datasets/test_degraded.tar` trên Google Drive 5TB  

---

### 1. QUY CHUẨN 6 BIẾN THỂ SUY BIẾN VẬT LÝ (DEGRADATION SPECIFICATIONS)
1. **Nén lượng tử JPEG (JPEG Compression)**:
   - `jpeg_q30`: Chất lượng $Q=30$ (Nén sâu, artifact khối $8 \times 8$ cường độ cao - Tử huyệt của CLIP)
   - `jpeg_q50`: Chất lượng $Q=50$ (Nén trung bình chuẩn chia sẻ web)
   - `jpeg_q70`: Chất lượng $Q=70$ (Nén nhẹ)
2. **Làm mờ Gauss (Gaussian Blur)**:
   - `blur_s1`: Bán kính mờ $\sigma = 1.0$
   - `blur_s2`: Bán kính mờ $\sigma = 2.0$
3. **Biến dạng mạng xã hội (Downsample-Upsample)**:
   - `down_up`: Thu nhỏ $224 \rightarrow 112$ rồi phóng to lại $224$ bằng nội suy Bicubic

---

### 2. TIÊU CHUẨN NGHIỆM THU (DEFINITION OF DONE - DoD):
- [ ] Dọn dẹp sạch tập Clean, đúng chính xác 18 tập chuẩn (82,003 ảnh, không trùng lặp).
- [ ] Script `tools/make_degraded.py` đa tiến trình chạy ổn định, không lỗi rò rỉ RAM.
- [ ] Bảo toàn $100\%$ cấu trúc thư mục nhị phân `0_real/` và `1_fake/` trên toàn bộ các tập con.
- [ ] Đóng gói cục bộ trên SSD NVMe trước khi sao chép sang Drive, bảo đảm tệp `test_degraded.tar` tồn tại vĩnh viễn trên Drive 5TB.

In [ ]:
# ================================================================================
# BƯỚC 1: KẾT NỐI GOOGLE DRIVE 5TB & ĐỒNG BỘ MÃ NGUỒN GIT TỪ NHÁNH HOANG
# ================================================================================
from google.colab import drive
import os, sys

print("[*] Đang kết nối Google Drive 5TB...")
drive.mount('/content/drive')
DRIVE_ROOT = "/content/drive/MyDrive/Fatformer"
assert os.path.exists(DRIVE_ROOT), f"LỖI: Không tìm thấy thư mục {DRIVE_ROOT}. Vui lòng kiểm tra lại cấu trúc Drive!"
print(f"[✓] Đã kết nối thành công kho lưu trữ: {DRIVE_ROOT}")

# Đồng bộ mã nguồn Git nhánh hoang
%cd /content
REPO_DIR = "/content/fatformer-xla"
if not os.path.exists(REPO_DIR):
    print("[*] Đang clone repository từ branch hoang...")
    !git clone -b hoang https://github.com/LeeVHoangtk3/fatformer-xla.git
else:
    print("[*] Repository đã tồn tại, đang fetch và checkout nhánh hoang mới nhất...")
    %cd {REPO_DIR}
    !git fetch origin hoang
    !git checkout hoang
    !git pull origin hoang

%cd {REPO_DIR}
print(f"[✓] Thư mục làm việc hiện tại: {os.getcwd()}")
!git log -n 1 --oneline

In [ ]:
# ================================================================================
# BƯỚC 2: GIẢI NÉN VÀ CHUẨN HÓA CẤU TRÚC 18 TẬP TEST CLEAN (ĐÚNG 82,003 ẢNH)
# ================================================================================
import os, shutil, tarfile, time

CLEAN_DIR = "/content/dataset_local/test_clean"
os.makedirs(CLEAN_DIR, exist_ok=True)

# 1. Giải nén 10 tập Diffusion (1.22 GB)
diff_tar = f"{DRIVE_ROOT}/datasets/test_benchmark_diffusion.tar"
if os.path.exists(diff_tar):
    print(f"[*] Đang giải nén 10 tập Diffusion từ {diff_tar}...")
    !tar -xf "{diff_tar}" -C "{CLEAN_DIR}"
else:
    print(f"[!] CẢNH BÁO: Không tìm thấy {diff_tar}")

# 2. Giải nén 8 tập GANs (18.74 GB)
gans_tar = f"{DRIVE_ROOT}/datasets/test_benchmark_gans.tar"
if os.path.exists(gans_tar):
    print(f"[*] Đang giải nén 8 tập GANs từ {gans_tar}...")
    !tar -xf "{gans_tar}" -C "{CLEAN_DIR}"
else:
    print(f"[!] CẢNH BÁO: Không tìm thấy {gans_tar}")

# 3. CHUẨN HÓA DÀN PHẲNG CẤU TRÚC: Chuyển toàn bộ thư mục con ra CLEAN_DIR và xóa thư mục cha thừa
print("\n[*] Đang chuẩn hóa cấu trúc thư mục con để loại bỏ trùng lặp...")

# Dàn phẳng diffusion_test/
diff_inner = os.path.join(CLEAN_DIR, "diffusion_test")
if os.path.exists(diff_inner):
    for item in os.listdir(diff_inner):
        src = os.path.join(diff_inner, item)
        dst = os.path.join(CLEAN_DIR, item)
        if not os.path.exists(dst):
            shutil.move(src, dst)
    shutil.rmtree(diff_inner, ignore_errors=True)

# Dàn phẳng test/ (nơi chứa 8 tập GANs nếu có)
test_inner = os.path.join(CLEAN_DIR, "test")
if os.path.exists(test_inner):
    for item in os.listdir(test_inner):
        src = os.path.join(test_inner, item)
        dst = os.path.join(CLEAN_DIR, item)
        if not os.path.exists(dst):
            shutil.move(src, dst)
    shutil.rmtree(test_inner, ignore_errors=True)

# Kiểm tra danh sách 18 subsets
subsets = sorted([d for d in os.listdir(CLEAN_DIR) if os.path.isdir(os.path.join(CLEAN_DIR, d))])
print(f"\n[✓] Đã chuẩn hóa thành công {len(subsets)}/18 tập test Clean:")
for s in subsets:
    print(f"  - {s}")

# Đếm tổng số ảnh Clean (kỳ vọng đúng 82,003 ảnh)
total_clean = sum(len(files) for _, _, files in os.walk(CLEAN_DIR) if any(f.lower().endswith(('.jpg','.jpeg','.png')) for f in files))
print(f"\n• Tổng số ảnh Clean thực tế: {total_clean:,} ảnh (Kỳ vọng: 82,003 ảnh)")

In [ ]:
# ================================================================================
# BƯỚC 3: KIỂM TRA MÃ NGUỒN TOOLS/MAKE_DEGRADED.PY & SỐ NHÂN CPU
# ================================================================================
import multiprocessing as mp

num_cpus = mp.cpu_count()
print(f"[*] Số CPU cores khả dụng trên máy ảo Colab: {num_cpus}")
!python tools/make_degraded.py --help

In [ ]:
# ================================================================================
# BƯỚC 4: CHẠY PIPELINE SINH 6 BIẾN THỂ SUY BIẾN ĐA TIẾN TRÌNH
# ================================================================================
# Tự động bỏ qua ảnh đã sinh hợp lệ (Safe Resume/Skip)
# Với 82,003 ảnh chuẩn, tổng cộng chỉ cần sinh 492,018 ảnh (tiết kiệm 50% thời gian!)

!python tools/make_degraded.py \
    --input_dir /content/dataset_local/test_clean \
    --output_root /content/dataset_local/test_degraded \
    --degradations all \
    --num_workers 4

In [ ]:
# ================================================================================
# BƯỚC 5: KIỂM ĐỊNH TÍNH TOÀN VẸN CỦA DỮ LIỆU ĐÃ SINH (AUDIT VERIFICATION)
# ================================================================================
import os
from pathlib import Path

DEG_ROOT = Path("/content/dataset_local/test_degraded")
EXPECTED_DEGS = ["jpeg_q30", "jpeg_q50", "jpeg_q70", "blur_s1", "blur_s2", "down_up"]

print("=" * 75)
print("KIỂM TOÁN TÍNH TOÀN VẸN BỘ KIỂM THỬ SUY BIẾN VẬT LÝ")
print("=" * 75)

if not DEG_ROOT.exists():
    print(f"[!] LỖI: Thư mục {DEG_ROOT} không tồn tại!")
else:
    for deg in EXPECTED_DEGS:
        deg_path = DEG_ROOT / deg
        if deg_path.exists():
            img_count = sum(1 for _ in deg_path.rglob("*.jpg"))
            print(f"• Biến thể [{deg:<10}]: {img_count:>8,d} ảnh (.jpg) -> [✓] SẴN SÀNG")
        else:
            print(f"• Biến thể [{deg:<10}]: CHƯA TỒN TẠI -> [!] THIẾU")
print("=" * 75)

In [ ]:
# ================================================================================
# BƯỚC 6: [GIAO THỨC CHỐNG LỖI FUSE] ĐÓNG GÓI CỤC BỘ TRƯỚC RỒI COPY SANG DRIVE 5TB
# ================================================================================
# NGUYÊN TẮC VÀNG: Không tar trực tiếp vào /content/drive/ vì FUSE buffer sẽ bị nghẽn,
# thay vào đó tar siêu tốc trên SSD NVMe /content/ rồi copy nguyên khối sang Drive!

import os, time, shutil
from google.colab import drive

LOCAL_TAR = "/content/test_degraded.tar"
DRIVE_TARGET = "/content/drive/MyDrive/Fatformer/datasets/test_degraded.tar"

# 1. Đóng gói cục bộ siêu tốc trên SSD NVMe local (~15-20 giây)
print(f"[1/3] Đang đóng gói nội bộ trên SSD NVMe: {LOCAL_TAR}...")
t0 = time.time()
!tar -cf {LOCAL_TAR} -C /content/dataset_local test_degraded

assert os.path.exists(LOCAL_TAR), "LỖI: Không tạo được tệp tar cục bộ!"
local_gb = os.path.getsize(LOCAL_TAR) / (1024 ** 3)
print(f"  -> [✓] Đóng gói SSD thành công trong {time.time() - t0:.1f}s | Dung lượng: {local_gb:.2f} GB")

# 2. Sao chép nguyên khối sang Google Drive 5TB
print(f"\n[2/3] Đang sao chép tệp nén sang Google Drive 5TB: {DRIVE_TARGET}...")
os.makedirs(os.path.dirname(DRIVE_TARGET), exist_ok=True)
t1 = time.time()
!cp {LOCAL_TAR} {DRIVE_TARGET}
print(f"  -> [✓] Sao chép sang Drive hoàn tất trong {time.time() - t1:.1f}s!")

# 3. Ép buộc Google Drive FUSE đồng bộ dữ liệu (Flush Cache)
print("\n[3/3] Đang ép bộ nhớ đệm Google Drive đồng bộ dữ liệu đám mây (Flush Cache)...")
drive.flush_and_unmount()
print("  -> [✓] Đã Flush hoàn tất. Đang mount lại để kiểm tra...")
drive.mount('/content/drive')

# 4. Kiểm tra nghiệm thu dứt điểm
if os.path.exists(DRIVE_TARGET):
    drive_gb = os.path.getsize(DRIVE_TARGET) / (1024 ** 3)
    print("\n" + "=" * 80)
    print(f"🎉 XÁC NHẬN NGHIỆM THU: TỆP NÉN TEST_DEGRADED.TAR ĐÃ NẰM VĨNH VIỄN TRÊN DRIVE 5TB!")
    print(f"• Đường dẫn:      {DRIVE_TARGET}")
    print(f"• Dung lượng:     {drive_gb:.2f} GB")
    print("• Trạng thái:     SẴN SÀNG 100% ĐỂ THỰC THI TASK 2.3 (BENCHMARK DEGRADED)!")
    print("=" * 80)
else:
    print(f"[!] LỖI: Tệp {DRIVE_TARGET} chưa xuất hiện trên Drive. Vui lòng kiểm tra lại quyền truy cập!")

In [ ]:
# ================================================================================
# BƯỚC 7: CELL KIỂM TRA ĐỘC LẬP TÍNH TOÀN VẸN HEADER TAR TRÊN GOOGLE DRIVE
# ================================================================================
import tarfile

DRIVE_TARGET = "/content/drive/MyDrive/Fatformer/datasets/test_degraded.tar"
print(f"[*] Kiểm tra header tệp TAR tại: {DRIVE_TARGET}")

if os.path.exists(DRIVE_TARGET):
    is_tar = tarfile.is_tarfile(DRIVE_TARGET)
    print(f"• Định dạng TAR hợp lệ: {is_tar}")
    with tarfile.open(DRIVE_TARGET, 'r') as tar:
        sample_files = [m.name for idx, m in enumerate(tar) if idx < 10]
    print("• Mẫu 10 phần tử đầu tiên bên trong file tar:")
    for s in sample_files:
        print(f"   - {s}")
    print("\n[✓] HOÀN TẤT KIỂM TRA TOÀN VẸN TASK 2.1!")
else:
    print("[!] Không tìm thấy tệp trên Drive!")